# Write a Custom ReLU Operator in C++ (Step 1 — CPU Forward)

**Difficulty**: 🔴 Hard

**Companies**: NVIDIA, Meta, Google, xAI

---

### Problem Statement

PyTorch ships hundreds of operators — but when you need something custom (a fused kernel, a novel activation, a proprietary algorithm), you write a **C++ extension**.

In Step 1, you'll implement a **CPU-only ReLU forward pass** in C++ and register it with PyTorch's dispatcher so you can call it from Python just like `torch.relu`.

ReLU is simple by design: $$ \text{ReLU}(x) = \max(0, x) $$ — so you focus on the **mechanics of the extension system**, not the math.

---

### What you'll learn

| Concept | What it is |
|---|---|
| **`TORCH_LIBRARY`** | Macro that registers your operator's **schema** (name + type signature) with PyTorch's dispatcher |
| **Dispatcher** | PyTorch's runtime system that routes `my_op(tensor)` → the correct kernel for that tensor's device/dtype |
| **`TORCH_LIBRARY_IMPL`** | Macro that binds a **dispatch key** (e.g. `CPU`) to your C++ kernel implementation |
| **Tensor API** | `torch::Tensor`, `data_ptr<T>()`, `where()`, `zeros_like()`, `numel()` — the C++ surface for manipulating tensors |
| **`PYBIND11_MODULE`** | Required entry point so `cpp_extension.load()` can import the compiled `.so` |

---

### Requirements

1. **C++ ReLU kernel** — Implement `relu_cpu_forward` in `relu_op.cpp`. Strategy A (`torch::where`) or Strategy B (manual loop).
2. **Schema registration** — Register the operator name and signature using `TORCH_LIBRARY`.
3. **CPU binding** — Bind your kernel to the `CPU` dispatch key using `TORCH_LIBRARY_IMPL`.
4. **Python entry point** — Uncomment the `PYBIND11_MODULE` block so the `.so` can be imported.
5. **Build & validate** — Compile with `torch.utils.cpp_extension.load` and verify against `F.relu`.

---

### Constraints

- ✅ CPU only (CUDA comes in Step 2)
- ✅ Must work with `float32` tensors of any shape
- ❌ Do **not** call `torch.relu` or `F.relu` inside your C++ code

---


In [1]:
import torch
import torch.nn.functional as F
from torch.utils.cpp_extension import load
import os

print(f"PyTorch version: {torch.__version__}")
print(f"Working directory: {os.getcwd()}")

CPP_FILE = "relu_op_SOLN.cpp"
print(f"Solution C++ source exists: {os.path.exists(CPP_FILE)}")


PyTorch version: 2.12.0+cu130
Working directory: /home/zireael/TorchLeet/torch/hard/custom-cpp-op
Solution C++ source exists: True


### Solution overview

The solution file is `relu_op_SOLN.cpp`. It contains 5 blocks:

| # | Block | Purpose |
|---|-------|---------|
| 1 | `relu_cpu_forward()` | The actual ReLU: `max(0, x)` via `torch::where` |
| 2 | `TORCH_LIBRARY(custom_relu, m)` | Registers schema `relu(Tensor) -> Tensor` |
| 3 | `TORCH_LIBRARY_IMPL(custom_relu, CPU, m)` | Binds CPU kernel |
| 3b | `TORCH_LIBRARY_IMPL(custom_relu, AutogradCPU, m)` | Autograd fallthrough (suppresses deprecation warning) |
| 4 | `PYBIND11_MODULE(TORCH_EXTENSION_NAME, m)` | Python entry point |

### Build and test

Run the cells below to compile and validate the solution.


In [2]:
# Build the solution
custom_relu = load(
    name="custom_relu_soln",
    sources=["relu_op_SOLN.cpp"],
    verbose=True,
)

print("\nSolution extension loaded!")
op = torch.ops.custom_relu.relu
print(f"Operator: {op}")
print(f"Schemas: {op._schemas}")


[1/2] c++ -MMD -MF relu_op_SOLN.o.d -DTORCH_EXTENSION_NAME=custom_relu_soln -DTORCH_API_INCLUDE_EXTENSION_H -isystem /home/zireael/TorchLeet/.venv/lib/python3.13/site-packages/torch/include -isystem /home/zireael/TorchLeet/.venv/lib/python3.13/site-packages/torch/include/torch/csrc/api/include -isystem /home/zireael/.local/share/uv/python/cpython-3.13.9-linux-x86_64-gnu/include/python3.13 -fPIC -std=c++20 -c /home/zireael/TorchLeet/torch/hard/custom-cpp-op/relu_op_SOLN.cpp -o relu_op_SOLN.o 
[2/2] c++ relu_op_SOLN.o -shared -L/home/zireael/TorchLeet/.venv/lib/python3.13/site-packages/torch/lib -lc10 -ltorch_cpu -ltorch -ltorch_python -o custom_relu_soln.so

Solution extension loaded!
Operator: custom_relu.relu
Schemas: {'': custom_relu::relu(Tensor input) -> Tensor}


In [3]:
# Test 1: Positive values → unchanged
x_pos = torch.tensor([1.0, 2.0, 3.5, 100.0])
out = torch.ops.custom_relu.relu(x_pos)
expected = F.relu(x_pos)
print(f"Positive values: {out.tolist()}")
assert torch.allclose(out, expected)
print("PASSED ✓")


Positive values: [1.0, 2.0, 3.5, 100.0]
PASSED ✓


In [4]:
# Test 2: Negative values → zero
x_neg = torch.tensor([-1.0, -2.0, -0.5, -100.0])
out = torch.ops.custom_relu.relu(x_neg)
expected = F.relu(x_neg)
print(f"Negative values: {out.tolist()}")
assert torch.allclose(out, expected)
print("PASSED ✓")


Negative values: [0.0, 0.0, 0.0, 0.0]
PASSED ✓


In [5]:
# Test 3: Mixed values
x_mixed = torch.tensor([-3.0, -1.0, 0.0, 1.0, 3.0])
out = torch.ops.custom_relu.relu(x_mixed)
expected = F.relu(x_mixed)
print(f"Mixed values: {out.tolist()}")
assert torch.allclose(out, expected)
print("PASSED ✓")


Mixed values: [0.0, 0.0, 0.0, 1.0, 3.0]
PASSED ✓


In [6]:
# Test 4: Zero handling
x_zero = torch.zeros(5)
out = torch.ops.custom_relu.relu(x_zero)
expected = F.relu(x_zero)
print(f"Zeros: {out.tolist()}")
assert torch.allclose(out, expected)
print("PASSED ✓")


Zeros: [0.0, 0.0, 0.0, 0.0, 0.0]
PASSED ✓


In [7]:
# Test 5: Multi-dimensional tensor
torch.manual_seed(42)
x_2d = torch.randn(4, 8)
out = torch.ops.custom_relu.relu(x_2d)
expected = F.relu(x_2d)
max_err = (out - expected).abs().max().item()
print(f"2D input shape: {x_2d.shape}, max error: {max_err:.2e}")
assert torch.allclose(out, expected)
print("PASSED ✓")


2D input shape: torch.Size([4, 8]), max error: 0.00e+00
PASSED ✓


In [8]:
# Test 6: Autograd
x_grad = torch.randn(3, 4, requires_grad=True)
out = torch.ops.custom_relu.relu(x_grad)
loss = out.sum()
loss.backward()

print(f"Input grad:\n{x_grad.grad}")
expected_grad = (x_grad > 0).float()
print(f"Expected grad:\n{expected_grad}")
assert torch.allclose(x_grad.grad, expected_grad)
print("PASSED ✓ — autograd flows correctly, no deprecation warning!")


Input grad:
tensor([[1., 1., 0., 0.],
        [1., 1., 0., 1.],
        [1., 1., 1., 0.]])
Expected grad:
tensor([[1., 1., 0., 0.],
        [1., 1., 0., 1.],
        [1., 1., 1., 0.]])
PASSED ✓ — autograd flows correctly, no deprecation warning!


### 🎉 All tests passed!

Below is the full solution code for reference.


In [9]:
# Display the completed solution
with open("relu_op_SOLN.cpp", "r") as f:
    solution_code = f.read()
print(solution_code)


#include <torch/extension.h>

// ============================================================================
// Step 1: Implement ReLU forward on CPU
// ============================================================================
// ReLU(x) = max(0, x) — applied element-wise to every value in the tensor.
//
// Strategy A (torch::where) — clean, delegates to PyTorch's own vectorized
// comparison and selection, so it benefits from internal optimizations.
// Strategy B (manual loop) — gives full control; useful when you need
// custom logic (e.g., clamping, mixed-precision handling, or fused
// operations) that torch::where can't express.

torch::Tensor relu_cpu_forward(const torch::Tensor& input) {
    // Strategy A:
    return torch::where(input > 0, input, torch::zeros_like(input));

    // Strategy B (equivalent, uncomment to use):
    // auto output = torch::empty_like(input);
    // auto in_ptr  = input.data_ptr<float>();
    // auto out_ptr = output.data_ptr<float>();
    // for 

### How `TORCH_LIBRARY` and the dispatcher work (diagram)

Here's the full picture of what happens when you call `torch.ops.custom_relu.relu(x)`:

```
Python:  torch.ops.custom_relu.relu(x)
              │
              ▼
┌─────────────────────────────────────────────────┐
│              PyTorch Dispatcher                  │
│                                                  │
│  1. Looks up "custom_relu::relu" in the         │
│     operator registry (populated by             │
│     TORCH_LIBRARY at .so load time).            │
│                                                  │
│  2. Reads the schema:                           │
│     "relu(Tensor input) -> Tensor"              │
│     → validates argument types & count.         │
│                                                  │
│  3. Checks the tensor's dispatch key:           │
│     CPU tensor → dispatch key = CPU             │
│     (CUDA tensor → CUDA, etc.)                  │
│                                                  │
│  4. Looks up the kernel for (op, dispatch key)  │
│     in the implementation registry              │
│     (populated by TORCH_LIBRARY_IMPL).          │
│                                                  │
│  5. Calls relu_cpu_forward(x).                  │
│                                                  │
│  6. Returns the result back to Python.          │
└─────────────────────────────────────────────────┘
              │
              ▼
Python:  result tensor
```

**Key insight:** The dispatcher is what enables PyTorch's "write once, run anywhere" model. Your Python code calls `custom_relu.relu(x)` regardless of whether `x` is on CPU or CUDA — the dispatcher picks the right kernel. You just haven't registered a CUDA kernel yet (that's Step 2!).

### Why the AutogradCPU fallthrough matters

When you define an operator with only a `CPU` kernel:

1. **Forward pass** — the dispatcher finds your `CPU` kernel ✓
2. **Backward pass** — PyTorch looks for an `AutogradCPU` kernel. If it finds a **fallthrough**, it uses automatic element-wise differentiation to derive the backward pass. If it finds nothing, it still works but shows a deprecation warning (PyTorch ≥ 2.x).

The solution registers `CppFunction::makeFallthrough()` on `AutogradCPU` to explicitly opt into automatic differentiation:

```cpp
TORCH_LIBRARY_IMPL(custom_relu, AutogradCPU, m) {
    m.impl("relu", torch::CppFunction::makeFallthrough());
}
```

For ReLU, the automatically derived backward is `grad_input = grad_output * (input > 0)`, which is correct. For operations where the automatic gradient would be wrong (e.g., a custom matmul with a specific backward formula), you'd register a **manual** `AutogradCPU` kernel with a custom `backward()` implementation — that's for a future step!

### Tensor API reference for future custom ops

```cpp
// --- Creation ---
torch::empty_like(t)          // uninitialized, same shape/dtype/device
torch::zeros_like(t)          // all zeros
torch::ones_like(t)           // all ones
torch::empty({M, N}, opts)    // uninitialized, explicit shape

// --- Element-wise ---
torch::where(cond, a, b)      // cond ? a : b
torch::clamp(t, min, max)     // clip values
torch::exp(t)                  // e^t
torch::log(t)                  // ln(t)
torch::sqrt(t)                 // sqrt(t)

// --- Reductions ---
t.sum()                        // sum of all elements
t.max()                        // max of all elements
torch::sum(t, dim)            // sum along dimension

// --- Access ---
t.data_ptr<float>()           // raw float* (CPU) — ensure dtype is float32 first
t.data_ptr<int64_t>()         // raw int64_t*
t.numel()                     // total number of elements
t.sizes()                     // IntArrayRef of shape
t.size(dim)                   // size of one dimension
t.dtype()                     // scalar type (e.g., torch::kFloat32)
t.device()                    // device (e.g., torch::kCPU)
```

### Dispatch key reference

| Key | When it's used |
|---|---|
| `CPU` | CPU tensors |
| `CUDA` | CUDA tensors |
| `AutogradCPU` | Backward pass for CPU tensors |
| `AutogradCUDA` | Backward pass for CUDA tensors |
| `CompositeImplicitAutograd` | Op works for all backends, autograd is automatic |
| `Meta` | Shape inference (no data) — used by `torch.compile` |
| `QuantizedCPU` | Quantized tensors on CPU |
